# Budget Forcing

**Paper**: [s1: Simple test-time scaling](https://arxiv.org/abs/2501.19393)

**Authors**: Niklas Muennighoff, Zitong Yang, Weijia Shi, Xiang Lisa Li, Li Fei-Fei, Hannaneh Hajishirzi, Luke Zettlemoyer, Percy Liang, Emmanuel Candes, Tatsunori Hashimoto

Budget forcing controls the length of a reasoning model's thinking at test time. It caps the thinking phase at a token budget and can either shorten reasoning (force the closing think tag once the budget is hit) or lengthen it (append an extension such as "Wait" to prompt continued reasoning) before generating the final answer.

Budget forcing is a decoding driver built on the generic phased driver: a bounded thinking phase, optional extension rounds, a forced closing tag, and an unbounded answer phase.

The method assumes a reasoning model. The thinking-phase boundary is the model's own closing think tag, so the driver can only find that boundary if the model actually emits one; on a non-reasoning model the tag never appears and the method degenerates to blind truncation plus a pasted-in tag.

## Method parameters

| parameter | type | description |
| --------- | ---- | ----------- |
| `max_thinking_tokens` | `int` | Token budget for each thinking segment |
| `extension_text` | `str` | Text appended to prolong reasoning |
| `num_extensions` | `int` | Number of extension rounds (0 disables) |
| `end_think` | `str` | The closing-think marker |

## Setup

If running this from a Google Colab notebook, uncomment the clone cell below. It is not necessary when running from a virtual environment where the package is already installed.

In [1]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360

The following authentication steps may be necessary to access any gated models (after being granted access by Hugging Face). Uncomment the following if you need to log in to the Hugging Face Hub:

In [2]:
# !pip install -q python-dotenv
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

## Example: dialing a reasoning model's thinking budget

We use `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`, a small open reasoning model. Its chat template opens the thinking block (the prompt ends with `<think>`), and the model closes it by emitting `</think>` before writing its final answer, so the driver's boundary marker occurs naturally in every generation. Following the model card we sample with temperature 0.6 and top-p 0.95 rather than decoding greedily, with a fixed seed so runs are comparable.

The budget dial only moves answer quality on problems at the edge of the model's ability. On easy problems the model recovers from any truncation by quietly re-deriving the solution inside the unbounded answer phase, so every budget lands on the right answer and the dial appears inert. We therefore work on a Level 5 problem from MATH-500 (the `HuggingFaceH4/MATH-500` subset of MATH; this is row `test/algebra/297.json`): find the product of the $y$-coordinates of all distinct solutions of $y=x^2-8$ and $y^2=-5x+44$. The derivation is long and unforgiving (square, assemble a quartic, factor it twice, apply the quadratic formula, and multiply four values including a conjugate pair), so partial reasoning does not degrade gracefully. The correct answer is 1736.


In [3]:
import re

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.algorithms.output_control.budget_forcing.control import BudgetForcing

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
END_THINK = "</think>"
SAMPLING = {"do_sample": True, "temperature": 0.6, "top_p": 0.95}

# level 5 problem from MATH-500 (HuggingFaceH4/MATH-500, row test/algebra/297.json); the answer is 1736
PROBLEM = (
    "Find the product of the $y$-coordinates of all the distinct solutions $(x,y)$ "
    "for the two equations $y=x^2-8$ and $y^2=-5x+44$."
)
ANSWER = "1736"


/dccstor/principled_ai/users/erikmiehling/AISteer360/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Full reasoning streams are long, so three small helpers keep the outputs readable: one splits a generation into its thinking span and final answer, one counts thinking tokens, and one pulls the final answer out of its `\boxed{...}` wrapper (falling back to the last number in the stream). The split is on the first closing tag, so whatever the model generates after the (possibly forced) tag counts as answer, and when a generation runs out of tokens before any tag appears, the whole stream counts as thinking.


In [4]:
def split_thinking(text: str, end_think: str = END_THINK) -> tuple[str, str]:
    if end_think in text:
        thinking, answer = text.split(end_think, 1)
        return thinking, answer.strip()
    return text, ""


def num_tokens(tokenizer, text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def extract_boxed(text: str) -> str:
    start = text.rfind("\\boxed{")
    if start != -1:  # brace-matched content of the last \boxed{...}
        i = start + len("\\boxed{")
        depth = 1
        for j in range(i, len(text)):
            depth += (text[j] == "{") - (text[j] == "}")
            if depth == 0:
                return text[i:j].strip()
    numbers = re.findall(r"-?\d+", text)
    return numbers[-1] if numbers else ""


### Baseline: the model's natural thinking length

First, how the model behaves unforced. We generate with a plain `model.generate` call and a generous token limit, then measure how long the model chooses to think. Per the model card's usage recommendation for math problems, the prompt asks for the final answer inside `\boxed{...}`, which is what the extractor above anchors on.


In [5]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

prompt = f"{PROBLEM} Please reason step by step, and put your final answer within \\boxed{{}}."
chat = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)

set_seed(42)
baseline_ids = model.generate(**inputs, max_new_tokens=6144, pad_token_id=tokenizer.eos_token_id, **SAMPLING)
baseline_text = tokenizer.decode(baseline_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

thinking, answer = split_thinking(baseline_text)
print(f"thinking tokens: {num_tokens(tokenizer, thinking)}")
print(f"extracted answer: {extract_boxed(baseline_text)} (target {ANSWER})")
print(f"\nend of answer: ...{answer[-350:]}")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/339 [00:00<04:37,  1.22it/s]

Loading weights:   1%|          | 2/339 [00:01<04:25,  1.27it/s]

Loading weights:   2%|▏         | 6/339 [00:01<01:07,  4.95it/s]

Loading weights:   5%|▌         | 17/339 [00:01<00:18, 17.06it/s]

...

Loading weights:  94%|█████████▍| 318/339 [00:08<00:00, 23.62it/s]

Loading weights:  96%|█████████▌| 325/339 [00:09<00:00, 28.95it/s]

Loading weights:  97%|█████████▋| 330/339 [00:09<00:00, 22.92it/s]

Loading weights: 100%|██████████| 339/339 [00:09<00:00, 36.04it/s]

thinking tokens: 3958
extracted answer: 1736 (target 1736)

end of answer: ...\( y^2 - y - 56 = 0 \) gives roots \( y = \frac{1 \pm 15}{2} \), which are 8 and -7.

The distinct \( y \)-coordinates are 8, -7, \(\frac{-1 + 5\sqrt{5}}{2}\), and \(\frac{-1 - 5\sqrt{5}}{2}\). The product of these roots is the constant term of the quartic equation, which is 1736.

Thus, the product of the \( y \)-coordinates is:
\[
\boxed{1736}
\]


Left alone, the model thinks for a long stretch on this problem before committing to an answer, several times longer than the few hundred tokens it spends on grade-school word problems. That natural length is the reference point for everything below: shortening means cutting below it, extending means pushing past where the model would have stopped. Note that at temperature 0.6 the natural length and even the final answer vary from run to run; the sweep at the end averages over seeds for exactly this reason.


### Shortening: cap the budget and force the tag

We cap thinking at `max_thinking_tokens=256` with no extensions. The plan is a thinking phase that stops at the closing tag or at 256 tokens (whichever comes first), the forced `</think>`, then the answer phase. At 256 tokens the model has barely finished setting up the quartic, so the thinking span below ends abruptly where the tag was pasted in. This is the "shorten" half of s1. From here on, each pipeline wraps the model loaded above (`SteeringPipeline` accepts a preloaded `model` and `tokenizer`), so nothing is re-downloaded between configurations.


In [6]:
budget_forcing = BudgetForcing(max_thinking_tokens=256, num_extensions=0, end_think=END_THINK)

pipeline = SteeringPipeline(model=model, tokenizer=tokenizer, controls=[budget_forcing])
pipeline.steer()

set_seed(42)
output = pipeline.generate(
    input_ids=inputs["input_ids"].to(pipeline.model.device),
    max_new_tokens=768,
    pad_token_id=tokenizer.eos_token_id,
    **SAMPLING,
)
forced_text = tokenizer.decode(output[0], skip_special_tokens=True)
thinking, answer = split_thinking(forced_text)

print(f"thinking tokens: {num_tokens(tokenizer, thinking)}")
print(f"extracted answer: {extract_boxed(forced_text)} (target {ANSWER})")
print(f"\nend of thinking span: ...{thinking[-160:]}")
print(f"\nend of answer: ...{answer[-350:]}")


thinking tokens: 256
extracted answer: 20 (target 1736)

end of thinking span: ...:

x⁴ - 16x² + 64 = -5x + 44

Hmm, I should bring all terms to one side to set the equation equal to zero. Let me subtract (-5x + 44) from both sides:

x⁴ - 16x

end of answer: ...      1 -1  -15 20  0

So, after division, the polynomial becomes:

(x + 1)(x³ - x² - 15x + 20) = 0

Now, let's factor the cubic equation x³ - x² - 15x + 20. Again, let's try the Rational Root Theorem. Possible roots are ±1, ±2, ±4, ±5, ±10, ±20.

Testing x = 1:

1 - 1 - 15 + 20 = 5 ≠ 0

x = 2:

8 - 4 - 30 + 20 = -6 ≠ 0

x = 4:

64 - 16 - 60 + 20 =


The thinking span stops mid-sentence at exactly the budget, and the model is forced to answer from whatever partial setup it has. It still tries to compensate: the "answer" it writes after the forced tag attempts to re-derive the whole solution from scratch. On easy problems that recovery succeeds and hides the cut entirely, which is precisely why easy problems make budget forcing look inert. Here the compressed re-derivation has to survive a quartic factorization and a conjugate-pair product without room to check itself, and it typically slips somewhere along the way, so the boxed answer usually comes out wrong. Cutting the budget below what the problem needs now costs accuracy rather than style.


### Extending: append "Wait" and keep thinking

Extensions are the "lengthen" half of s1. Each extension round appends `Wait` to the stream and opens another bounded thinking segment, so a thought the budget would have cut short gets prolonged instead. We keep the per-segment budget at 512 tokens so the splice points are easy to locate: the driver appends `Wait` right after tokens 512 and 1025 of the continuation.


In [7]:
budget_forcing = BudgetForcing(
    max_thinking_tokens=512,
    extension_text="Wait",
    num_extensions=2,
    end_think=END_THINK,
)

pipeline = SteeringPipeline(model=model, tokenizer=tokenizer, controls=[budget_forcing])
pipeline.steer()

set_seed(42)
output = pipeline.generate(
    input_ids=inputs["input_ids"].to(pipeline.model.device),
    max_new_tokens=2048,
    pad_token_id=tokenizer.eos_token_id,
    **SAMPLING,
)
extended_text = tokenizer.decode(output[0], skip_special_tokens=True)
thinking, answer = split_thinking(extended_text)
print(f"thinking tokens: {num_tokens(tokenizer, thinking)}")
print(f"extracted answer: {extract_boxed(extended_text)} (target {ANSWER})")

wait_len = num_tokens(tokenizer, "Wait")
out_ids = output[0]
for i, splice_at in enumerate([512, 512 + wait_len + 512], start=1):
    window = tokenizer.decode(out_ids[splice_at - 20:splice_at + wait_len + 20], skip_special_tokens=True)
    print(f"\nsplice {i}: ...{window}...")

print(f"\nend of answer: ...{answer[-350:]}")


thinking tokens: 1538
extracted answer: 20 (target 1736)

splice 1: ... Vieta's relates the coefficients of a polynomial to sums and products of its roots.

But in thisWait, let's think about it step by step.

We have the quartic equation: x⁴...

splice 2: ..._i and y_i. Maybe another approach.

Wait, perhaps I can express the quartic equation inWait, the quartic equation is x⁴ - 16x² + 5x +...

end of answer: ...t of y_i's is the product of (x_i² - 8) for each root. However, this approach is complex. Instead, by considering the quartic equation and recognizing that y = x² - 8, we can express the product of y_i's in terms of the quartic's coefficients. The product of the y-coordinates is 20.

**Answer:** The product of the y-coordinates is 20.

$\boxed{20}$


Each splice shows the same pattern: the segment is cut mid-thought at its budget, the appended `Wait` lands, and the model picks the reasoning back up, often by re-examining what it had just concluded. The total thinking length is now set by the driver, not by when the model felt done. Three bounded segments give the model roughly 1.5k thinking tokens here, which is real progress on the derivation but often still short of what this problem takes; the sweep below puts numbers on that.


### The s1 story: answer quality vs. thinking budget

Budget forcing is the mechanism behind s1's test-time scaling curves, where answer quality is a function of allotted thinking compute. On a problem past the model's comfortable range the dial shows up in accuracy itself. The sweep below runs the same problem at three budgets with four sampled runs each, and tabulates the average thinking tokens actually used, how many runs land on 1736, and the extracted answers themselves. The twelve runs share the one model already in memory and take a few minutes on a GPU.


In [8]:
budgets = [256, 1024, 4096]
num_trials = 4

results = []
for budget in budgets:
    sweep_pipeline = SteeringPipeline(
        model=model,
        tokenizer=tokenizer,
        controls=[BudgetForcing(max_thinking_tokens=budget, num_extensions=0, end_think=END_THINK)],
    )
    sweep_pipeline.steer()
    trials = []
    for seed in range(num_trials):
        set_seed(seed)
        output = sweep_pipeline.generate(
            input_ids=inputs["input_ids"].to(sweep_pipeline.model.device),
            max_new_tokens=max(budget, 768),
            pad_token_id=tokenizer.eos_token_id,
            **SAMPLING,
        )
        text = tokenizer.decode(output[0], skip_special_tokens=True)
        thinking, _ = split_thinking(text)
        trials.append((num_tokens(tokenizer, thinking), extract_boxed(text)))
    results.append((budget, trials))

print(f"{'budget':>7}  {'thinking tokens':>16}  {'correct':>9}  answers")
for budget, trials in results:
    mean_used = round(sum(used for used, _ in trials) / num_trials)
    correct = sum(answer == ANSWER for _, answer in trials)
    answers = ", ".join(answer if answer else "-" for _, answer in trials)
    print(f"{budget:>7}  {mean_used:>16}  {f'{correct}/{num_trials}':>9}  {answers}")


 budget   thinking tokens    correct  answers
    256               256        0/4  8, 8, -56, 1
   1024              1024        1/4  770, 1736, -56, 2
   4096              4096        4/4  1736, 1736, 1736, 1736


The thinking-token column tracks the budget, which is the compute half of the s1 curve. The correct column is the other half. At the smallest budget the model answers from a truncated setup, and the compressed recovery in the answer phase rarely survives the full derivation; at the largest budget the derivation usually completes inside the thinking span and lands on 1736, with the middle budget transitional. The exact counts move from run to run at temperature 0.6, but the upward trend with budget is the stable part, and it is the s1 result in miniature: one integer trades decode compute against correctness. Note that this readout depends on the problem sitting at the model's edge. Rows the model finds easy are flat at the top of the curve no matter the budget (the answer phase absorbs the cut), and rows past its reach are flat at the bottom; any MATH-500 Level 5 row with a long derivation and a short numeric answer can play the same role if this one drifts out of that band on a different model variant.


### Mechanics

`BudgetForcing` is a preset of the generic phased driver. Its per-example plan is:

- `Generated(until="</think>", budget=max_thinking_tokens)`, the bounded thinking phase
- `num_extensions` repetitions of `Fixed(extension_text)` followed by another bounded `Generated`
- `Fixed("</think>")`, the forced closing tag
- `Generated()`, the unbounded answer phase

Phase boundaries are substring stops on the closing marker only; the opening `<think>` plays no role in the mechanics (here it lives in the prompt, courtesy of the chat template). `Fixed` phases are plain token appends, so when the model closes its thinking naturally within budget, the forced tag still lands and the stream carries the tag twice. Plans are built per example, and batched inputs are handled by looping over rows. Every `Generated` phase delegates to `model.generate` with the pipeline's composed stacks, so a step-level control (for example RAD) steers each phase, including the extensions.

### Takeaway

Budget forcing turns thinking length into an inference-time dial: one integer trades answer quality against decode compute, and the "Wait" trick buys extra reasoning on demand without touching weights or prompts. It only makes sense on models that already externalize their reasoning between think tags.

[phased_decoding.ipynb](../generics/phased_decoding.ipynb) demonstrates the generic this preset is built on, including a thinking-intervention plan that splices steering text into the reasoning stream rather than bounding its length. See the [output control](https://ibm.github.io/AISteer360/concepts/controls/#output-control) section of the docs for the full family.